# Analyze Drudge hyperlinks

In [48]:
import re
import sys
from pathlib import Path
from urllib.parse import urlparse

In [185]:
import tldextract
import storysniffer

In [50]:
import numpy as np
import pandas as pd
import altair as alt

In [51]:
this_dir = Path("__file__").parent.absolute()
sys.path.append(this_dir.parent)
sys.path.append(str(this_dir.parent / "newshomepages"))

In [52]:
import altair_theme

In [53]:
alt.themes.register('palewire', altair_theme.theme)
alt.themes.enable('palewire')

ThemeRegistry.enable('palewire')

In [54]:
extracts_dir = this_dir.parent / "extracts" / "csv"

In [55]:
analysis_dir = this_dir.parent / "_analysis"

Read in the sample data

In [56]:
df = pd.read_csv(
    extracts_dir / "drudge-hyperlinks-sample.csv",
    usecols=[
        'handle',
        'file_name',
        'date',
        'text',
        'url',
    ],
    dtype=str,
    parse_dates=["date"]
)

Guess links with `storysniffer`

In [256]:
links_df = df.groupby(["text", "url"]).agg({
    "handle": "size",
    "date": "min"
}).rename(columns={"handle": "n", "date": "earlier_date"}).reset_index()

In [257]:
links_df

,text,url,n,earlier_date
0,\nGolfer set to become first trans to earn LPG...,https://www.dailymail.co.uk/news/article-11129...,2,2022-08-20
1,\nGolfer to become first trans to earn LPGA to...,https://www.dailymail.co.uk/news/article-11129...,4,2022-08-21
2,\nPeople Who Do Strength Training Live Longer ...,https://dnyuz.com/2022/08/24/people-who-do-str...,2,2022-08-25
3,\nWildfire near Yosemite National Park explode...,https://apnews.com/article/wildfires-fires-cal...,3,2022-07-24
4,Committee Kicks Off Capitol Attack Public Pro...,https://apnews.com/article/capitol-siege-ivank...,1,2022-06-09
...,...,...,...,...
7089,[UK] INDEPENDENT,http://www.independent.co.uk/,258,2022-06-03
7090,[UK] METRO,https://metro.co.uk/,258,2022-06-03
7091,[UK] SUN,http://www.thesun.co.uk,258,2022-06-03
7092,[UK] TELEGRAPH,http://www.telegraph.co.uk/,258,2022-06-03


In [58]:
sniffer = storysniffer.StorySniffer()

In [59]:
links_df['is_story'] = links_df.apply(lambda x: sniffer.guess(x['url'], text=x['text']), axis=1)

In [60]:
links_df.is_story.value_counts()

True     6835
False     259
Name: is_story, dtype: int64

In [61]:
links_df.is_story.value_counts(normalize=True)

True     0.96349
False    0.03651
Name: is_story, dtype: float64

In [62]:
links_df.to_csv("./drudge-hyperlinks-storysniffer-guesses.csv", index=False)

Make some manual fixes

In [63]:
blacklist = [
    "/privacy/",
]

In [64]:
links_df.loc[
    links_df.url.isin(blacklist),
    'is_story'
] = False

In [173]:
correction_list = [
    "\.(substack|theankler|commonsense|thedispatch).(com|news)/p/",
    "^https://time.com/\d{5,}/*",
    "^https://*.studyfinds.org/*.{5,}",
    "^https://*.bbc.com/news/*.{5,}",
    "^https://www.jpost.com/breaking-news/*.{5,}",
    "^https://www.jpost.com/[a-z]{5,}/*.{5,}",
    "^https://*.braintomorrow.com/*.{5,}"
    "^https://finance.yahoo.com/news/*.{5,}",
    "^https://www.vice.com/en/article/*.{5,}",
    "^https://news.yahoo.com/*.{5,}",
]

In [174]:
for c in correction_list:
    links_df.loc[links_df.url.str.contains(c, regex=True), 'is_story'] = True

/tmp/ipykernel_64114/1150215247.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  links_df.loc[links_df.url.str.contains(c, regex=True), 'is_story'] = True


Knock out anything that appears most of the time

In [175]:
n = len(df.file_name.unique())

In [176]:
too_much = links_df.n >= n * .5

In [177]:
links_df.loc[too_much, 'is_story'] = False

Knock out anything with a bad URL

In [221]:
links_df.loc[~links_df.url.str.startswith("http"), 'is_story'] = False

In [222]:
links_df[
    (links_df.n < 40) &
    (~links_df.is_story)
].sort_values("url").head(5)

,text,url,n,is_story,domain
4453,Orlando to become 'physical center of the meta...,https://www.the-sun.com/tech/5661470/orlando-...,1,False,https.
4452,Orlando to become 'physical center of metavers...,https://www.the-sun.com/tech/5661470/orlando-...,4,False,https.
6169,Tiger dies after contracting at Ohio zoo...,/https://www.cbsnews.com/news/tiger-dies-after...,1,False,.
4703,Poll numbers reveal trouble for Biden beyond t...,How bad things are for Biden,1,False,How bad things are for Biden.
3450,LIVE: TEMP MAP,http://hp2.wright-weather.com/icons/us_heat.gif,2,False,wright-weather.com


In [223]:
links_df.to_csv("./drudge-hyperlinks-storysniffer-tweaks.csv", index=False)

In [224]:
story_df = links_df[links_df.is_story].copy()

In [225]:
story_df['domain'] = story_df.url.apply(lambda x: f"{tldextract.extract(x).domain}.{tldextract.extract(x).suffix}")

In [247]:
story_df

,text,url,n,is_story,domain,is_trump
0,\nGolfer set to become first trans to earn LPG...,https://www.dailymail.co.uk/news/article-11129...,2,True,dailymail.co.uk,False
1,\nGolfer to become first trans to earn LPGA to...,https://www.dailymail.co.uk/news/article-11129...,4,True,dailymail.co.uk,False
2,\nPeople Who Do Strength Training Live Longer ...,https://dnyuz.com/2022/08/24/people-who-do-str...,2,True,dnyuz.com,False
3,\nWildfire near Yosemite National Park explode...,https://apnews.com/article/wildfires-fires-cal...,3,True,apnews.com,False
4,Committee Kicks Off Capitol Attack Public Pro...,https://apnews.com/article/capitol-siege-ivank...,1,True,apnews.com,True
...,...,...,...,...,...,...
7079,Zelensky warns of 'dangerous' situation as sid...,https://www.nbcnews.com/news/world/russia-ukra...,3,True,nbcnews.com,False
7080,Zelenskyy losing 60 to 100 soldiers a day...,https://www.businessinsider.com/ukraine-losing...,1,True,businessinsider.com,False
7081,Zuckerberg Sells San Fran Home for Record $31 ...,https://nypost.com/2022/07/25/mark-zuckerberg-...,1,True,nypost.com,False
7082,Zuckerberg designing clothes to wear in metave...,https://www.the-sun.com/tech/5599222/mark-zuck...,6,True,the-sun.com,False


Tally domains

In [226]:
domain_df = story_df.groupby(["domain"]).size().rename("n").reset_index().sort_values("n", ascending=False)

In [242]:
domain_df['percent'] = (domain_df.n / domain_df.n.sum()) * 100

In [245]:
domain_df.head(10)

,domain,n,percent
182,msn.com,806,11.744135
356,yahoo.com,589,8.582253
17,apnews.com,507,7.387440
352,wsj.com,372,5.420370
84,dnyuz.com,310,4.516975
73,dailymail.co.uk,297,4.327554
66,cnn.com,218,3.176453
65,cnbc.com,216,3.147312
284,the-sun.com,211,3.074457
212,nypost.com,207,3.016174


In [229]:
def is_trump(row):
    token_list = [t.lower() for t in row['text'].split()]
    if 'trump' in token_list:
        return True
    elif 'donald' in token_list:
        return True
    elif 'don' in token_list:
        return True
    else:
        if 'trump' in row['url']:
            return True
    return False

In [230]:
story_df['is_trump'] = story_df.apply(is_trump, axis=1)

In [231]:
story_df.is_trump.value_counts()

False    6213
True      650
Name: is_trump, dtype: int64

In [232]:
trump_df = story_df[story_df.is_trump].copy()

In [233]:
date_df = df[['url', 'date']].sort_values(["url", "date"], ascending=[True, True]).drop_duplicates("url", keep="first")

In [234]:
trump_df = trump_df.merge(date_df, on="url").sort_values("date", ascending=False)

In [235]:
trump_df.head()

,text,url,n,is_story,domain,is_trump,date
183,Documents at Mar-a-Lago Were Moved and Hidden ...,https://news.yahoo.com/u-justice-dept-responds...,1,True,yahoo.com,True,2022-08-31
175,Demands to Be Reinstated as President...,https://www.mediaite.com/trump/trump-re-ups-hi...,1,True,mediaite.com,True,2022-08-31
36,'Those Records Do Not Belong to Him'...,https://dnyuz.com/2022/08/30/justice-departmen...,1,True,dnyuz.com,True,2022-08-31
532,TRUMP CAUGHT HOARDING NATION'S SECRETS,https://apnews.com/article/mar-a-lago-governme...,1,True,apnews.com,True,2022-08-31
505,Secret Service agent for Trump retires after J...,https://www.yahoo.com/news/secret-official-cen...,3,True,yahoo.com,True,2022-08-30


In [236]:
list(trump_df.text.unique())[:30]

['Documents at Mar-a-Lago Were Moved and Hidden as Feds Sought Them...',
 'Demands to Be Reinstated as President...',
 "'Those Records Do Not Belong to Him'...",
 "TRUMP CAUGHT HOARDING NATION'S SECRETS",
 'Secret Service agent for Trump retires after Jan. 6 scrutiny...',
 'Republicans forced into strained defense...',
 'Shares barrage of QAnon content...',
 'Trump boasted  he had intelligence about Macron sex life...',
 'The Don  boasted  he had intelligence about Macron sex life...',
 "What's Behind War With Intel Community?",
 'Legal team advances broad view of presidential powers in Mar-a-Lago probe...',
 'INDICTMENT WATCH INTENSIFIES',
 'MORE WARNINGS OF UNREST',
 "'TRUTH' barred from GOOGLE store over moderation concerns...",
 'MUST BE HELD IMMEDIATELY!',
 'DEMENTING DON DEMANDS NEW ELECTION?',
 'Legal team advances broad view of presidential powers at Mar-a-Lago...',
 'THE DON CALLS FOR UPRISING!',
 'Ben Shapiro Blames Trump...',
 'Lindsey Graham predicts riots if prosecuted...'